# CRANE-LLM Demo

A typical Kaggle-style notebook: load bike-sharing trips, clean them, explore, and fit a few regression models. It is based on a real Kaggle notebook and contains crashes that notebooks like it really hit.

Use it to try CRANE-LLM: run the cells from the top, and before running a cell, select it and click **CRANE-LLM** in the toolbar to find out whether it will crash.

The purple boxes mark six scenes. Each fix is commented in its cell, after `# Fix`.

In [ ]:
# Once per machine: the API key of the model CRANE-LLM asks.
# Not needed for the built-in checks, which run without a model.
# import crane_llm
# crane_llm.set_api_key("sk-...")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 200)
%matplotlib inline

In [ ]:
data = pd.read_csv("data/data_test.csv", index_col=0).sample(frac=1, random_state=0)
data.head()

## Cleaning

In [ ]:
data = data[data['Haversine'] != 0].reset_index(drop=True)

data['Distance'] = data['Distance'].abs()

data = data[data['Distance'] != 0].reset_index(drop=True)
data.shape

<div style="background:#f3e8ff; border-left:6px solid #7c3aed; border-radius:6px; padding:8px 14px; color:#1f1f1f;">
<b style="color:#6d28d9;">Scene 1 &middot; Built-in check: crash; + origin</b><br>
Run this cell, then check <code>data0.describe()</code> below.<br>Expect: red, <i>certain</i> <code>AttributeError</code>; the origin points back to this cell.
</div>

In [ ]:
data0 = data.dropna(inplace=True)  # Fix: data.dropna(inplace=True), without "data0 =": it returns None

In [ ]:
data0.describe()  # Fix: data.describe()

## Exploration

<div style="background:#f3e8ff; border-left:6px solid #7c3aed; border-radius:6px; padding:8px 14px; color:#1f1f1f;">
<b style="color:#6d28d9;">Scene 2 &middot; LLM prediction: crash</b><br>
Check this cell.<br>Expect: red, <i>LLM prediction</i>: <code>data1</code> is a <code>Categorical</code>, not a DataFrame.
</div>

In [ ]:
data1 = pd.Categorical(data['PDweek'], categories=range(7), ordered=True)
data1.cat.rename_categories(['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'], inplace=True)
# Fix: replace the two lines above with
# days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# data1 = data.assign(DayOfWeek=pd.Categorical.from_codes(data['PDweek'], days, ordered=True))

average_duration = data1.groupby('DayOfWeek')['Duration'].mean()  # Fix: groupby('DayOfWeek', observed=True)

plt.plot(average_duration.index, average_duration.values)
plt.xlabel('Day of Week')
plt.ylabel('Average Duration')
plt.title('Average Trip Duration by Day of Week')
plt.xticks(rotation=45)
plt.show()

<div style="background:#f3e8ff; border-left:6px solid #7c3aed; border-radius:6px; padding:8px 14px; color:#1f1f1f;">
<b style="color:#6d28d9;">Scene 3 &middot; LLM prediction: no crash</b><br>
Check this cell.<br>Expect: green.
</div>

In [ ]:
average_duration = data.groupby('Phour')['Duration'].mean()

plt.plot(average_duration.index, average_duration.values)
plt.xlabel('Hour of the Day')
plt.ylabel('Average Trip Duration')
plt.title('Average Trip Duration by Hour of the Day')
plt.xticks(range(24))
plt.show()

<div style="background:#f3e8ff; border-left:6px solid #7c3aed; border-radius:6px; padding:8px 14px; color:#1f1f1f;">
<b style="color:#6d28d9;">Scene 4 &middot; Runtime information on: crash; off: no crash</b><br>
Check this cell, then switch <i>Include runtime information</i> off and check again.<br>Expect: red <code>KeyError</code> (the column is <code>Humid</code>); without runtime information, usually green.
</div>

In [ ]:
weather_columns = ['Temp', 'GroundTemp', 'Humidity', 'Wind', 'Solar', 'Precip', 'Snow', 'Dust']  # Fix: 'Humidity' -> 'Humid'
for column in weather_columns:
    print(f"{column:>10}: {data[column].corr(data['Duration']):+.3f}")

## Modelling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data.drop(columns=['Duration']), data['Duration'], test_size=0.25, random_state=43)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
def rmse(y_true, predictions):
    return mean_squared_error(y_true, predictions) ** 0.5

def get_metrics(y_true, predictions):
    print("=" * 50)
    print(f"MAE: {mean_absolute_error(y_true, predictions)}")
    print(f"MSE: {mean_squared_error(y_true, predictions)}")
    print(f"RMSE: {rmse(y_true, predictions)}")
    print(f"R2 Score: {r2_score(y_true, predictions)}")
    print("=" * 50)

### Linear regression

In [ ]:
scaler = StandardScaler()
scaled_X_train = scaler.fit_transform(X_train)
scaled_X_test = scaler.transform(X_test)

In [ ]:
lrg = LinearRegression()
lrg.fit(scaled_X_train, y_train)

lrg_preds = lrg.predict(scaled_X_test)
get_metrics(y_test, lrg_preds)

### A new feature: rush hour

<div style="background:#f3e8ff; border-left:6px solid #7c3aed; border-radius:6px; padding:8px 14px; color:#1f1f1f;">
<b style="color:#6d28d9;">Scene 5 &middot; Built-in check: crash</b><br>
Run this cell, then check the cell below.<br>Expect: red (model fitted before the data changed), <i>certain</i> <code>ValueError</code> (new column <code>RushHour</code>); the origin lists this cell and the scaler cell.
</div>

In [ ]:
rush_hours = [7, 8, 9, 17, 18, 19]
X_train = X_train.assign(RushHour=X_train['Phour'].isin(rush_hours).astype(int))
X_test = X_test.assign(RushHour=X_test['Phour'].isin(rush_hours).astype(int))
X_train.shape, X_test.shape

In [ ]:
# Fix: fit the scaler and the model again on the new columns first
# scaler = StandardScaler()
# lrg = LinearRegression().fit(scaler.fit_transform(X_train), y_train)
lrg_preds = lrg.predict(scaler.transform(X_test))
get_metrics(y_test, lrg_preds)

### Random forest

<div style="background:#f3e8ff; border-left:6px solid #7c3aed; border-radius:6px; padding:8px 14px; color:#1f1f1f;">
<b style="color:#6d28d9;">Scene 6 &middot; Check cells before they run: crash (by built-in checker)</b><br>
Switch <i>Check cells before they run</i> on and run it.<br>Expect: stopped at once with <code>CrashPrevented</code>.
<br>Run this cell directly: it trains for ~30 s, then crashes.
</div>

In [ ]:
# crane: run
rf_model = RandomForestRegressor(n_estimators=200, random_state=24)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
get_metrics(y_test, rf_pred)  # Fix: rf_preds